In [1]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config


c:\Users\jonat\anaconda3\envs\reddit_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [3]:
# set seed
random.seed(42)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# User-selected number of each 
num_topics = len(topic_cols)  
num_stocks = 0

# Randomly sample (without replacement), limited by available count
selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# Final filtered dataframe
X = X[selected_topics + selected_stocks]

# Convert stock returns to log returns: log(1+r)
X[selected_stocks] = np.log(X[selected_stocks] + 1)

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

## Grid Search

In [ ]:
# Objective Function
def objective_function(row):
    weight = 2 * (norm.cdf(abs(row['kappa_tstat'])) - 0.5)
    return row['r2_insample_stage2'] * weight
# -----------------------------
# Stopping criteria (NEW)
# -----------------------------
max_iterations = 20          # maximum refinement iterations
improvement_tol = 1e-6       # stop if best objective improves by less than this

# Initial grid
window_sizes = [200,300,400]
n_lags = [1,2,3,4,5]
lambda_base = 0.0001
lambda_values = [lambda_base * factor for factor in [0.9, 1.0, 1.1]]

# IMPORTANT: match grid_search expectations: window_sizes, n_lags, lambdas
current_param_grid = {
    'window_sizes': window_sizes,
    'n_lags': n_lags,
    'lambdas': lambda_values
}

results_all_rounds = []

prev_best_objective = -float('inf')

# ----------------------------------------
# loop 
# ----------------------------------------
for i in range(max_iterations):

    print(f"\nStarting refinement iteration {i+1}")

    # Run grid search
    summary_df, coefficients_df = grid_search(
        X, y, current_param_grid, verbose=True
    )

    # Evaluate objective
    summary_df['objective'] = summary_df.apply(objective_function, axis=1)

    # Track results
    results_all_rounds.append(summary_df.copy())

    # Pick best point of this iteration
    best_idx = summary_df['objective'].idxmax()
    best_row = summary_df.loc[best_idx]
    best_objective = float(best_row['objective'])

    print("Best this iteration:")
    print(best_row[['window_size', 'n_lags', 'lambda', 'objective']])

    # Early stopping checks
    if not np.isfinite(best_objective):
        print("Stopping: best objective is not finite (all candidates failed constraints).")
        break

    if i > 0:
        improvement = best_objective - prev_best_objective
        print(f"Improvement vs previous best: {improvement:.6g}")

        if improvement < improvement_tol:
            print(f"Stopping: improvement {improvement:.6g} < tolerance {improvement_tol:.6g}.")
            break

    prev_best_objective = best_objective

    # ----------------------------------------
    # Build new grid around best point
    # ----------------------------------------
    best_window = int(best_row['window_size'])
    best_n_lags = int(best_row['n_lags'])
    best_lambda = float(best_row['lambda'])

    # shrink window search range
    window_sizes_refined = list(range(best_window - 25, best_window + 26, 5))
    window_sizes_refined = [w for w in window_sizes_refined if w > 20]

    # shrink n_lags search range
    n_lags_refined = list(range(best_n_lags - 1, best_n_lags + 2))
    n_lags_refined = [l for l in n_lags_refined if l >= 1]

    # shrink lambda range 
    lambda_values_refined = np.linspace(
        0.8 * best_lambda,
        1.2 * best_lambda,
        num=5
    )

    # update grid for next iteration
    current_param_grid = {
        'window_sizes': window_sizes_refined,
        'n_lags': n_lags_refined,
        'lambdas': lambda_values_refined
    }

# ----------------------------------------
# Final evaluation
final_df = pd.concat(results_all_rounds, ignore_index=True)
final_df['objective'] = final_df.apply(objective_function, axis=1)
best_overall = final_df.loc[final_df['objective'].idxmax()]

print("\nBest hyperparameters after refinement:")
print(best_overall[['window_size', 'n_lags', 'lambda', 'objective']])

In [ ]:
# print for the 5 rows with the highest objective values r2 insample_stage2 and stage1, kappa ,kappa_tstat, lambda, window_size, n_lags
top_10 = final_df.nlargest(30, 'objective')
print(top_10[['r2_insample_stage2', 'r2_insample_stage1', 'r2_oos_stage2', 'kappa', 'kappa_tstat', 'lambda', 'window_size', 'n_lags']])

## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [ ]:
import optuna
import numpy as np

def objective(trial):

    window_size = trial.suggest_int("window_size", 10, 500, step=10)
    n_lags = trial.suggest_int("n_lags", 1, 15)
    lam = trial.suggest_float("lambda", 1e-5, 1e-1, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam, 
        standardize=True, verbose=False, return_details=False
    )
    
    summary = res.get("summary", {}) if res is not None else {}
    
    r2 = summary.get("r2_oos_stage2", np.nan)
    kappa = summary.get("kappa", np.nan)

    trial.set_user_attr("r2_raw", r2)
    trial.set_user_attr("kappa_raw", kappa)

    if np.isnan(r2) or np.isnan(kappa) or r2 < 0 or kappa < 0:
        return -1.0 # Hard penalty

    score = np.sqrt(r2 * kappa)
    
    return score

sampler = optuna.samplers.TPESampler(seed=42) 
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=300, n_jobs=4)

print("Best Geometric Mean Score:", study.best_value)
print("Best Params:", study.best_params)

best_trial = study.best_trial
print(f"Composed of R2: {best_trial.user_attrs['r2_raw']:.4f} and Kappa: {best_trial.user_attrs['kappa_raw']:.4f}")

[I 2026-01-13 16:40:41,560] A new study created in memory with name: no-name-70a5b817-5a5b-440c-8654-d0f9e9b3798c
[I 2026-01-13 16:41:18,711] Trial 3 finished with value: -1.0 and parameters: {'window_size': 260, 'n_lags': 10, 'lambda': 0.021708182906033466}. Best is trial 3 with value: -1.0.
[I 2026-01-13 16:41:39,114] Trial 2 finished with value: -1.0 and parameters: {'window_size': 470, 'n_lags': 13, 'lambda': 0.0037532143109315144}. Best is trial 3 with value: -1.0.
[I 2026-01-13 16:41:54,381] Trial 5 finished with value: -1.0 and parameters: {'window_size': 290, 'n_lags': 3, 'lambda': 0.0012423366291479116}. Best is trial 3 with value: -1.0.
[I 2026-01-13 16:42:23,470] Trial 4 finished with value: -1.0 and parameters: {'window_size': 480, 'n_lags': 8, 'lambda': 0.00025697523143200865}. Best is trial 3 with value: -1.0.
[I 2026-01-13 16:42:47,374] Trial 7 finished with value: -1.0 and parameters: {'window_size': 150, 'n_lags': 11, 'lambda': 0.06094426773991621}. Best is trial 3 wit

In [ ]:
res  = estimate_single_config(
    X, y,
    window_size= 289,
    n_lags=4,
    lambda_val=0.002799646010728981,
    standardize=True,   
    verbose=True,
    return_details=True
)